# Harmonic ILC $y$-map with pyILC

Reconstruct a Compton-$y$ map from the coadded skies

$$
T_\nu = T_\mathrm{CMB} + \Delta T_\mathrm{tSZ}(\nu) + \Delta T_\mathrm{CIB}(\nu)
\quad\text{at}\quad \nu \in \{100,143,353\}\,\mathrm{GHz}.
$$

Uses **harmonic ILC** (`wavelet_type: TopHatHarmonic`) preserving the tSZ SED
(output in dimensionless Compton-$y$), following the pyILC example API in
`software_packages/pyilc/notebooks/Planck_CMB_HILC.ipynb`.

| Input | Path |
|-------|------|
| Coadds (K_CMB) | `maps_100_143_353/coadd/sky_*_K.fits` |
| Truth $y$ | `maps_100_143_353/raw/compton_y_nside4096.fits` |
| ILC output | `maps_100_143_353/ilc_output/` |
| YAML config | `maps_100_143_353/hilc_y_100_143_353.yml` |

> **Kernel**: `cosmo_env`. Needs substantial RAM at $N_\mathrm{side}=4096$.


In [1]:
from pathlib import Path

import healpy as hp
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

from pyilc.input import ILCInfo
from pyilc.wavelets import Wavelets, harmonic_ILC

MAP_ROOT = Path(
    "/home/ext_andyxlcnb_gmail_com/flamingo_mock_analysis/maps_100_143_353"
)
RAW_DIR = MAP_ROOT / "raw"
COADD_DIR = MAP_ROOT / "coadd"
OUT_DIR = MAP_ROOT / "ilc_output"
YAML_FILE = MAP_ROOT / "hilc_y_100_143_353.yml"

FREQS = [100, 143, 353]
NSIDE = 4096

OUT_DIR.mkdir(parents=True, exist_ok=True)
assert YAML_FILE.is_file(), YAML_FILE
for nu in FREQS:
    p = COADD_DIR / f"sky_CMB_tSZ_CIB_{nu}GHz_nside{NSIDE}_K.fits"
    assert p.is_file(), p

print("config:", YAML_FILE)
print("output:", OUT_DIR)


config: /home/ext_andyxlcnb_gmail_com/flamingo_mock_analysis/maps_90_150_353/hilc_y_90_150_353.yml
output: /home/ext_andyxlcnb_gmail_com/flamingo_mock_analysis/maps_90_150_353/ilc_output


## 1. Peek at coadded inputs and truth $y$

In [2]:
y_true = hp.read_map(str(RAW_DIR / f"compton_y_nside{NSIDE}.fits"), dtype=np.float64)
print(
    f"truth y: mean={y_true.mean():.3e}, std={y_true.std():.3e}, "
    f"Nside={hp.get_nside(y_true)}"
)

fig = plt.figure(figsize=(12, 3.2))
for i, nu in enumerate(FREQS, start=1):
    m = hp.read_map(
        str(COADD_DIR / f"sky_CMB_tSZ_CIB_{nu}GHz_nside{NSIDE}_uK.fits"),
        dtype=np.float64,
    )
    vmax = np.percentile(np.abs(m), 99)
    hp.mollview(
        m,
        title=rf"coadd ${nu}$ GHz",
        unit=r"$\mu\mathrm{K}$",
        min=-vmax,
        max=vmax,
        sub=(1, 3, i),
        hold=True,
    )
plt.show()

truth y: mean=1.611e-06, std=2.070e-06, Nside=4096


## 2. Load YAML → `ILCInfo` and run harmonic ILC

Same driver pattern as the upstream Planck HILC notebook (`read_maps` → `maps2alms` → `alms2cls` → `harmonic_ILC`).

`ILC_preserved_comp: tSZ` → output map is in Compton-$y$ units.

In [3]:
def run_hilc(info: ILCInfo) -> Path:
    print("reading maps / bandpasses / beams...")
    info.read_maps()
    info.read_bandpasses()
    info.read_beams()

    print(
        f"HILC: N_freqs={info.N_freqs}, ELLMAX={info.ELLMAX}, "
        f"N_scales={info.N_scales}, BinSize-driven ell bins"
    )
    wv = Wavelets(
        N_scales=info.N_scales,
        ELLMAX=info.ELLMAX,
        tol=1.0e-6,
        taper_width=info.taper_width,
    )
    assert info.wavelet_type == "TopHatHarmonic"
    wv.TopHatHarmonic(info.ellbins)

    if not info.weights_exist:
        print("maps → alms...")
        info.maps2alms()
        print("alms → C_ell...")
        info.alms2cls()

    print("harmonic_ILC...")
    harmonic_ILC(wv, info, resp_tol=info.resp_tol, map_images=False)

    out = Path(
        info.output_dir
        + info.output_prefix
        + "needletILCmap_component_"
        + info.ILC_preserved_comp
        + info.output_suffix
        + ".fits"
    )
    print("saved:", out)
    return out


info = ILCInfo(str(YAML_FILE))
info.output_dir = str(OUT_DIR) + "/"
# ilc_path = run_hilc(info)

reading maps / bandpasses / beams...
reading in map 0
No units found in header; assuming K_CMB.
reading in map 1
No units found in header; assuming K_CMB.
reading in map 2
No units found in header; assuming K_CMB.
HILC: N_freqs=3, ELLMAX=4096, N_scales=81, BinSize-driven ell bins
maps → alms...
alms → C_ell...
harmonic_ILC...
doing main ILC!!
weight vector already exists: /home/ext_andyxlcnb_gmail_com/flamingo_mock_analysis/maps_90_150_353/ilc_output/yang26_weightvector_scale0_component_tSZ.txt
weight vector already exists: /home/ext_andyxlcnb_gmail_com/flamingo_mock_analysis/maps_90_150_353/ilc_output/yang26_weightvector_scale1_component_tSZ.txt
weight vector already exists: /home/ext_andyxlcnb_gmail_com/flamingo_mock_analysis/maps_90_150_353/ilc_output/yang26_weightvector_scale2_component_tSZ.txt
weight vector already exists: /home/ext_andyxlcnb_gmail_com/flamingo_mock_analysis/maps_90_150_353/ilc_output/yang26_weightvector_scale3_component_tSZ.txt
weight vector already exists: /home

OSError: File /home/ext_andyxlcnb_gmail_com/flamingo_mock_analysis/maps_90_150_353/ilc_output/yang26_needletILCmap_component_tSZ_hilc_y_90_150_353.fits already exists. If you mean to replace it then use the argument "overwrite=True".

## 3. Compare ILC $y$ to truth

Smooth both maps to the common ILC beam (`perform_ILC_at_beam = 5'`) before residuals / spectra.

In [4]:
# Discover output if cell above already ran previously
candidates = sorted(OUT_DIR.glob("*needletILCmap_component_tSZ*.fits"))
assert candidates, f"No ILC map in {OUT_DIR} — run the HILC cell first"
ilc_path = candidates[-1]
print("ILC map:", ilc_path)

y_ilc = hp.read_map(str(ilc_path), dtype=np.float64)
y_true = hp.read_map(str(RAW_DIR / f"compton_y_nside{NSIDE}.fits"), dtype=np.float64)

FWHM_ARCMIN = 5.0
y_true_sm = hp.smoothing(y_true, fwhm=np.radians(FWHM_ARCMIN / 60.0))
y_ilc_sm = hp.smoothing(y_ilc, fwhm=np.radians(FWHM_ARCMIN / 60.0))
resid = y_ilc_sm - y_true_sm

print(
    f"ILC  std={y_ilc_sm.std():.3e} | truth std={y_true_sm.std():.3e} | "
    f"resid std={resid.std():.3e}"
)
corr = np.corrcoef(y_ilc_sm, y_true_sm)[0, 1]
print(f"pixel correlation (smoothed): {corr:.4f}")

ILC map: /home/ext_andyxlcnb_gmail_com/flamingo_mock_analysis/maps_90_150_353/ilc_output/yang26_needletILCmap_component_tSZ_hilc_y_90_150_353.fits
ILC  std=1.058e-06 | truth std=1.239e-06 | resid std=3.744e-07
pixel correlation (smoothed): 0.9589


In [5]:
mpl.rcParams.update({"figure.dpi": 120, "savefig.dpi": 200})

vmax = np.percentile(np.abs(y_true_sm), 99)
rmax = np.percentile(np.abs(resid), 99)

fig = plt.figure(figsize=(12, 3.6))
hp.mollview(y_true_sm, title=r"truth $y$ (5')", unit="y", min=-vmax, max=vmax, sub=(1, 3, 1), hold=True)
hp.mollview(y_ilc_sm, title=r"HILC $y$ (5')", unit="y", min=-vmax, max=vmax, sub=(1, 3, 2), hold=True)
hp.mollview(resid, title=r"HILC $-$ truth", unit="y", min=-rmax, max=rmax, sub=(1, 3, 3), hold=True)
fig = plt.gcf()
fig.savefig(OUT_DIR / "hilc_y_vs_truth_mollview.png", bbox_inches="tight")
plt.show()

In [6]:
LMAX_SPEC = 3000
cl_tt = hp.anafast(y_true_sm, lmax=LMAX_SPEC)
cl_ii = hp.anafast(y_ilc_sm, lmax=LMAX_SPEC)
cl_ti = hp.anafast(y_true_sm, y_ilc_sm, lmax=LMAX_SPEC)
ell = np.arange(LMAX_SPEC + 1)
fac = ell * (ell + 1) / (2 * np.pi)
r_ell = np.divide(cl_ti, np.sqrt(cl_tt * cl_ii), out=np.zeros_like(cl_ti), where=(cl_tt > 0) & (cl_ii > 0))


def bin_cl(cl, delta=20):
    nb = (cl.size - 2) // delta
    eb = np.empty(nb)
    cb = np.empty(nb)
    for i in range(nb):
        lo, hi = 2 + i * delta, 2 + (i + 1) * delta
        eb[i] = ell[lo:hi].mean()
        cb[i] = cl[lo:hi].mean()
    return eb, cb


e_t, c_t = bin_cl(fac * cl_tt)
e_i, c_i = bin_cl(fac * cl_ii)
e_r, c_r = bin_cl(r_ell)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
ax = axes[0]
ax.loglog(e_t, c_t, color="k", lw=1.6, label=r"truth $y$")
ax.loglog(e_i, c_i, color="#D55E00", lw=1.6, label=r"HILC $y$")
ax.set_xlabel(r"$\ell$")
ax.set_ylabel(r"$\ell(\ell+1)C_\ell^{yy}/2\pi$")
ax.set_title(r"$y$ auto-spectrum (smoothed 5')")
ax.legend(frameon=False)

ax = axes[1]
ax.axhline(1.0, color="0.5", lw=0.8, ls="--")
ax.semilogx(e_r, c_r, color="#0072B2", lw=1.6)
ax.set_xlabel(r"$\ell$")
ax.set_ylabel(r"$r_\ell = C_\ell^{\mathrm{t}\times\mathrm{i}} / \sqrt{C_\ell^{tt}C_\ell^{ii}}$")
ax.set_ylim(0, 1.05)
ax.set_title(r"cross-correlation coeff.")
fig.tight_layout()
fig.savefig(OUT_DIR / "hilc_y_vs_truth_spectra.png", bbox_inches="tight")
plt.show()

## Notes

1. Inputs are coadds of lensed CMB + tSZ + CIB; CIB at 100/143 GHz uses the SED-scaling
   construction from `simulate_cib_frequency_maps.ipynb` (approximate).
2. No instrumental noise; beams are identical ($1'$) and ILC is performed at $5'$.
3. For a constrained ILC (e.g. deproject CMB), set in the YAML:
   `N_deproj: 1`, `ILC_deproj_comps: ['CMB']` and change `output_suffix`.
4. CLI alternative (after the pyILC patches in this env):
   `pyilc maps_100_143_353/hilc_y_100_143_353.yml`


## 4. $C_\ell^{yy}$: ground truth vs HILC (beam-deconvolved)

Uses the smoothed maps from §3 (`y_true_sm`, `y_ilc_sm`), then divides by
$B_\ell^2$ for the common Gaussian beam (`FWHM_ARCMIN`).

In [7]:
LMAX = 3000
DELTA_ELL = 20
# Cut where B_ell is tiny to avoid noisy deconvolution
B_ELL_MIN = 0.2

# Spectra of beam-convolved maps (ILC output + truth smoothed to same FWHM)
cl_yy_true_beam = hp.anafast(y_true_sm, lmax=LMAX)
cl_yy_ilc_beam = hp.anafast(y_ilc_sm, lmax=LMAX)

ell = np.arange(LMAX + 1)
bl = hp.gauss_beam(np.radians(FWHM_ARCMIN / 60.0), lmax=LMAX)
bl2 = bl**2

# Beam-deconvolved C_ell^{yy}
good = bl2 > B_ELL_MIN**2
cl_yy_true = np.full_like(cl_yy_true_beam, np.nan)
cl_yy_ilc = np.full_like(cl_yy_ilc_beam, np.nan)
cl_yy_true[good] = cl_yy_true_beam[good] / bl2[good]
cl_yy_ilc[good] = cl_yy_ilc_beam[good] / bl2[good]

dl_yy_true = ell * (ell + 1) * cl_yy_true / (2 * np.pi)
dl_yy_ilc = ell * (ell + 1) * cl_yy_ilc / (2 * np.pi)


def bin_spectrum(y, delta_ell=DELTA_ELL, lmin=2):
    n_bins = (y.size - lmin) // delta_ell
    ell_b, y_b = [], []
    for i in range(n_bins):
        lo = lmin + i * delta_ell
        hi = lo + delta_ell
        sl = y[lo:hi]
        if np.any(~np.isfinite(sl)):
            continue
        ell_b.append(ell[lo:hi].mean())
        y_b.append(np.nanmean(sl))
    return np.asarray(ell_b), np.asarray(y_b)


ell_b, dl_true_b = bin_spectrum(dl_yy_true)
_, dl_ilc_b = bin_spectrum(dl_yy_ilc)

fig, axes = plt.subplots(
    2, 1, figsize=(7.0, 5.5), sharex=True,
    gridspec_kw={"height_ratios": [2.2, 1.0], "hspace": 0.06},
)

ax = axes[0]
ax.loglog(ell_b, dl_true_b, color="k", lw=1.8, label=r"truth $C_\ell^{yy}$ (deconv.)")
ax.loglog(ell_b, dl_ilc_b, color="#D55E00", lw=1.8, label=r"HILC $C_\ell^{yy}$ (deconv.)")
ax.set_ylabel(r"$\ell(\ell+1)C_\ell^{yy}/(2\pi)$")
ax.legend(frameon=False, loc="upper right")
ax.set_title(rf"$yy$ power (beam-deconvolved, $B_\ell$ from {FWHM_ARCMIN:.0f}')")

ax = axes[1]
ax.axhline(1.0, color="0.5", lw=0.9, ls="--")
ax.semilogx(ell_b, dl_ilc_b / dl_true_b, color="#0072B2", lw=1.6)
ax.set_xlabel(r"Multipole $\ell$")
ax.set_ylabel(r"$C_\ell^{\mathrm{ILC}} / C_\ell^{\mathrm{true}}$")
ax.set_ylim(0.0, 2.0)

fig.savefig(OUT_DIR / "hilc_yy_power_truth_vs_ilc_deconv.png", bbox_inches="tight")
plt.show()
